In [1]:
import numpy as np
import pandas as pd
import time

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

TRAIN_PATH = "segment_alerts_all_airports_train.csv"
VALID_PATH = "dataset_set.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_valid = pd.read_csv(VALID_PATH)

for df_ in [df_train, df_valid]:
    if "Unnamed: 0" in df_.columns:
        df_.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
    df_["date"] = pd.to_datetime(df_["date"], utc=True)
    df_["icloud"] = df_["icloud"].astype(bool)

print("Train shape:", df_train.shape)
print("Valid shape:", df_valid.shape)

def month_to_season(m):
    if m in [12, 1, 2]:
        return "winter"
    elif m in [3, 4, 5]:
        return "spring"
    elif m in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"


def add_base_columns(df, strong_cg_threshold=None):
    df = df.copy()

    df["is_cg"] = (~df["icloud"]).astype("int8")
    df["is_ic"] = df["icloud"].astype("int8")

    df["is_0_3"] = (df["dist"] < 3).astype("int8")
    df["is_3_10"] = ((df["dist"] >= 3) & (df["dist"] < 10)).astype("int8")
    df["is_10_20"] = ((df["dist"] >= 10) & (df["dist"] < 20)).astype("int8")
    df["is_20_30"] = ((df["dist"] >= 20) & (df["dist"] < 30)).astype("int8")

    df["amp_abs"] = df["amplitude"].abs()

    if strong_cg_threshold is None:
        strong_cg_threshold = df.loc[df["is_cg"] == 1, "amp_abs"].quantile(0.90)

    df["is_strong_cg"] = (
        (df["is_cg"] == 1) & (df["amp_abs"] >= strong_cg_threshold)
    ).astype("int8")

    df["hour"] = df["date"].dt.hour
    df["month"] = df["date"].dt.month
    df["dayofyear"] = df["date"].dt.dayofyear
    df["weekday"] = df["date"].dt.weekday

    df["season"] = df["month"].apply(month_to_season)
    df["is_summer"] = (df["season"] == "summer").astype("int8")
    df["is_peak_hour"] = df["hour"].between(12, 18).astype("int8")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    az_rad = np.deg2rad(df["azimuth"])
    df["azimuth_sin"] = np.sin(az_rad)
    df["azimuth_cos"] = np.cos(az_rad)

    az_bins = [0, 45, 90, 135, 180, 225, 270, 315, 360]
    az_labels = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    df["az_sector"] = pd.cut(
        df["azimuth"].mod(360),
        bins=az_bins,
        labels=az_labels,
        include_lowest=True,
        right=False
    )

    df["cg_amp_abs"] = np.where(df["is_cg"] == 1, df["amp_abs"], np.nan)

    return df, strong_cg_threshold


def build_timestamp_features(df_all):
    base = (
        df_all.groupby(["airport", "date"], as_index=False)
        .agg(
            n_total=("lightning_id", "size"),
            n_cg=("is_cg", "sum"),
            n_ic=("is_ic", "sum"),
            n_strong_cg=("is_strong_cg", "sum"),

            n_0_3=("is_0_3", "sum"),
            n_3_10=("is_3_10", "sum"),
            n_10_20=("is_10_20", "sum"),
            n_20_30=("is_20_30", "sum"),

            dist_min=("dist", "min"),
            dist_mean=("dist", "mean"),
            dist_max=("dist", "max"),

            amp_abs_mean=("amp_abs", "mean"),
            amp_abs_max=("amp_abs", "max"),
            amp_abs_sum=("amp_abs", "sum"),

            cg_amp_abs_mean=("cg_amp_abs", "mean"),
            cg_amp_abs_max=("cg_amp_abs", "max"),

            maxis_mean=("maxis", "mean"),
            azimuth_mean=("azimuth", "mean"),
            azimuth_sin_mean=("azimuth_sin", "mean"),
            azimuth_cos_mean=("azimuth_cos", "mean"),

            hour=("hour", "first"),
            month=("month", "first"),
            dayofyear=("dayofyear", "first"),
            weekday=("weekday", "first"),
            season=("season", "first"),
            is_summer=("is_summer", "first"),
            is_peak_hour=("is_peak_hour", "first"),
            hour_sin=("hour_sin", "first"),
            hour_cos=("hour_cos", "first"),
            month_sin=("month_sin", "first"),
            month_cos=("month_cos", "first"),
        )
        .sort_values(["airport", "date"])
        .reset_index(drop=True)
    )

    base["cg_amp_abs_mean"] = base["cg_amp_abs_mean"].fillna(0.0)
    base["cg_amp_abs_max"] = base["cg_amp_abs_max"].fillna(0.0)

    base["cg_ratio_now"] = np.where(base["n_total"] > 0, base["n_cg"] / base["n_total"], 0.0)
    base["strong_cg_ratio_now"] = np.where(base["n_cg"] > 0, base["n_strong_cg"] / base["n_cg"], 0.0)

    sector_counts = (
        df_all.groupby(["airport", "date", "az_sector"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    sector_cols = [c for c in sector_counts.columns if c not in ["airport", "date"]]
    sector_counts = sector_counts.rename(columns={c: f"sector_{c}_count" for c in sector_cols})

    base = base.merge(sector_counts, on=["airport", "date"], how="left")

    for c in base.columns:
        if c.startswith("sector_") and c.endswith("_count"):
            base[c] = base[c].fillna(0)

    return base


def add_since_features(base):
    parts = []

    for airport, g in base.groupby("airport"):
        g = g.sort_values("date").copy()

        g["delta_prev_min"] = g["date"].diff().dt.total_seconds().div(60).fillna(9999)

        def since_last(col_name, new_name):
            last_date = g["date"].where(g[col_name] > 0).ffill().shift()
            g[new_name] = (g["date"] - last_date).dt.total_seconds().div(60).fillna(9999)

        since_last("n_0_3", "since_last_0_3_min")
        since_last("n_3_10", "since_last_3_10_min")
        since_last("n_10_20", "since_last_10_20_min")
        since_last("n_20_30", "since_last_20_30_min")
        since_last("n_cg", "since_last_cg_min")
        since_last("n_strong_cg", "since_last_strong_cg_min")

        parts.append(g)

    return pd.concat(parts, ignore_index=True)


def add_rolling_features(base_time):
    windows = ["5min", "10min", "30min"]

    count_cols = [
        "n_total", "n_cg", "n_ic", "n_strong_cg",
        "n_0_3", "n_3_10", "n_10_20", "n_20_30"
    ]
    sector_cols = [c for c in base_time.columns if c.startswith("sector_") and c.endswith("_count")]
    count_cols_all = count_cols + sector_cols

    mean_cols = [
        "dist_min", "dist_mean", "dist_max",
        "amp_abs_mean", "amp_abs_max", "amp_abs_sum",
        "cg_amp_abs_mean", "cg_amp_abs_max",
        "cg_ratio_now", "strong_cg_ratio_now"
    ]

    parts = []

    for airport, g in base_time.groupby("airport"):
        g = g.sort_values("date").copy().set_index("date")
        out = g.copy()

        for w in windows:
            roll_sum = g[count_cols_all].rolling(w, closed="both").sum()
            roll_sum.columns = [f"{c}__sum_{w}" for c in roll_sum.columns]

            roll_mean = g[mean_cols].rolling(w, closed="both").mean()
            roll_mean.columns = [f"{c}__mean_{w}" for c in roll_mean.columns]

            out = pd.concat([out, roll_sum, roll_mean], axis=1)

        out = out.reset_index()
        parts.append(out)

    return pd.concat(parts, ignore_index=True)


def add_derived_features(features_all):
    eps = 1e-6
    df = features_all.copy()

    df["trend_total_5_vs_10"] = df["n_total__sum_5min"] / (df["n_total__sum_10min"] + eps)
    df["trend_0_3_5_vs_10"] = df["n_0_3__sum_5min"] / (df["n_0_3__sum_10min"] + eps)
    df["trend_10_20_5_vs_10"] = df["n_10_20__sum_5min"] / (df["n_10_20__sum_10min"] + eps)
    df["trend_20_30_5_vs_10"] = df["n_20_30__sum_5min"] / (df["n_20_30__sum_10min"] + eps)
    df["trend_cg_5_vs_10"] = df["n_cg__sum_5min"] / (df["n_cg__sum_10min"] + eps)

    df["far_vs_near_5min"] = (df["n_20_30__sum_5min"] + 1) / (df["n_0_3__sum_5min"] + df["n_3_10__sum_5min"] + 1)
    df["far_vs_near_10min"] = (df["n_20_30__sum_10min"] + 1) / (df["n_0_3__sum_10min"] + df["n_3_10__sum_10min"] + 1)
    df["far_vs_mid_10min"] = (df["n_20_30__sum_10min"] + 1) / (df["n_10_20__sum_10min"] + 1)
    df["mid_vs_near_10min"] = (df["n_10_20__sum_10min"] + 1) / (df["n_0_3__sum_10min"] + df["n_3_10__sum_10min"] + 1)

    df["dist_range_now"] = df["dist_max"] - df["dist_min"]
    df["dist_range_10min"] = df["dist_max__mean_10min"] - df["dist_min__mean_10min"]
    df["dist_drift_5_10"] = df["dist_mean__mean_5min"] - df["dist_mean__mean_10min"]

    df["severity_now"] = (
        0.4 * np.tanh(df["amp_abs_mean"] / 20.0) +
        0.4 * df["cg_ratio_now"] +
        0.2 * (1 - np.clip(df["dist_mean"] / 30.0, 0, 1))
    )

    for w in ["5min", "10min", "30min"]:
        df[f"severity_{w}"] = (
            0.4 * np.tanh(df[f"amp_abs_mean__mean_{w}"] / 20.0) +
            0.4 * df[f"cg_ratio_now__mean_{w}"] +
            0.2 * (1 - np.clip(df[f"dist_mean__mean_{w}"] / 30.0, 0, 1))
        )

    df["strong_cg_ratio_10min"] = df["n_strong_cg__sum_10min"] / (df["n_cg__sum_10min"] + eps)
    df["strong_cg_ratio_30min"] = df["n_strong_cg__sum_30min"] / (df["n_cg__sum_30min"] + eps)

    return df

def compute_target_safe_to_end(df_feat, danger_dist_km=20):
    labeled = df_feat[df_feat["airport_alert_id"].notna()].copy()

    rows = []
    for (airport, alert_id), g in labeled.groupby(["airport", "airport_alert_id"]):
        g = g.sort_values("date").copy()
        times = g[["date"]].drop_duplicates().sort_values("date").copy()
        t = times["date"].to_numpy()

        dangerous_times = np.array(sorted(g.loc[g["dist"] < danger_dist_km, "date"].unique()))
        idx = np.searchsorted(dangerous_times, t, side="right")

        times["safe_to_end_now"] = (idx == len(dangerous_times)).astype("int8")
        times["airport"] = airport
        times["airport_alert_id"] = alert_id
        rows.append(times)

    return pd.concat(rows, ignore_index=True), labeled

def compute_alert_life(df_with_alerts):
    tmp = df_with_alerts[df_with_alerts["airport_alert_id"].notna()].copy()
    tmp = tmp.sort_values(["airport", "airport_alert_id", "date"]).copy()

    life_parts = []
    for (airport, alert_id), g in tmp.groupby(["airport", "airport_alert_id"]):
        g = g.sort_values("date").copy()

        g["elapsed_since_alert_start_min"] = (g["date"] - g["date"].iloc[0]).dt.total_seconds().div(60)
        g["cum_lightnings"] = np.arange(1, len(g) + 1)
        g["cum_cg"] = g["is_cg"].cumsum()
        g["cum_ic"] = g["is_ic"].cumsum()
        g["cum_strong_cg"] = g["is_strong_cg"].cumsum()

        g["cum_avg_rate"] = g["cum_lightnings"] / (g["elapsed_since_alert_start_min"] + 1)
        g["cum_cg_rate"] = g["cum_cg"] / (g["elapsed_since_alert_start_min"] + 1)
        g["cum_strong_cg_rate"] = g["cum_strong_cg"] / (g["elapsed_since_alert_start_min"] + 1)

        g_ts = (
            g.groupby(["airport", "airport_alert_id", "date"], as_index=False)
            .agg(
                elapsed_since_alert_start_min=("elapsed_since_alert_start_min", "max"),
                cum_lightnings=("cum_lightnings", "max"),
                cum_cg=("cum_cg", "max"),
                cum_ic=("cum_ic", "max"),
                cum_strong_cg=("cum_strong_cg", "max"),
                cum_avg_rate=("cum_avg_rate", "max"),
                cum_cg_rate=("cum_cg_rate", "max"),
                cum_strong_cg_rate=("cum_strong_cg_rate", "max"),
            )
        )

        life_parts.append(g_ts)

    return pd.concat(life_parts, ignore_index=True)


def compute_airport_static(df_train_feat):
    labeled = df_train_feat[df_train_feat["airport_alert_id"].notna()].copy()

    alerts_stats = (
        labeled.groupby(["airport", "airport_alert_id"], as_index=False)
        .agg(
            alert_start=("date", "min"),
            alert_end=("date", "max"),
            n_lightnings_alert=("lightning_id", "count"),
            n_cg_alert=("is_cg", "sum"),
            n_ic_alert=("is_ic", "sum"),
            mean_amp_alert=("amp_abs", "mean"),
        )
    )

    alerts_stats["alert_duration_min"] = (
        alerts_stats["alert_end"] - alerts_stats["alert_start"]
    ).dt.total_seconds().div(60)

    airport_static = (
        alerts_stats.groupby("airport", as_index=False)
        .agg(
            airport_alert_duration_mean=("alert_duration_min", "mean"),
            airport_alert_duration_median=("alert_duration_min", "median"),
            airport_alert_n_lightnings_mean=("n_lightnings_alert", "mean"),
            airport_alert_n_lightnings_median=("n_lightnings_alert", "median"),
            airport_alert_cg_mean=("n_cg_alert", "mean"),
            airport_alert_ic_mean=("n_ic_alert", "mean"),
            airport_alert_amp_mean=("mean_amp_alert", "mean"),
        )
    )

    return airport_static

Train shape: (507071, 13)
Valid shape: (183945, 13)


In [2]:
import numpy as np
import pandas as pd


def make_purged_time_folds(
    df,
    n_splits=5,
    date_col="date",
    group_col="airport_alert_id",
):
    """
    Crée des folds temporels par blocs de dates, avec purge des groupes
    (alertes) partagés entre train et validation.

    Principe :
    - on trie par date
    - on découpe les dates uniques en n_splits + 1 blocs
    - à chaque fold :
        train = tous les blocs précédents
        valid = bloc courant
    - puis on retire du train toutes les alertes présentes dans valid

    Retour :
    - liste de tuples (train_idx, valid_idx)
    """
    df = df.copy()
    df = df[df[date_col].notna()].sort_values(date_col).reset_index(drop=True)

    unique_dates = np.array(sorted(df[date_col].unique()))
    n_dates = len(unique_dates)

    if n_dates < n_splits + 1:
        raise ValueError(
            f"Pas assez de dates uniques ({n_dates}) pour {n_splits} splits."
        )

    date_blocks = np.array_split(unique_dates, n_splits + 1)
    folds = []

    for i in range(1, len(date_blocks)):
        train_dates = np.concatenate(date_blocks[:i])
        valid_dates = date_blocks[i]

        train_mask = df[date_col].isin(train_dates)
        valid_mask = df[date_col].isin(valid_dates)

        train_idx = df.index[train_mask]
        valid_idx = df.index[valid_mask]

        # Purge des alertes vues dans la validation
        if group_col in df.columns:
            valid_groups = set(df.loc[valid_idx, group_col].dropna().unique().tolist())

            if len(valid_groups) > 0:
                purged_train_mask = train_mask & (~df[group_col].isin(valid_groups))
                train_idx = df.index[purged_train_mask]

        if len(train_idx) == 0 or len(valid_idx) == 0:
            continue

        folds.append((train_idx.to_numpy(), valid_idx.to_numpy()))

    return folds

In [3]:
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from lightgbm import LGBMClassifier


def temporal_purged_cv_hyperparam_search(
    model_df_train,
    target_col="safe_to_end_now",
    date_col="date",
    group_col="airport_alert_id",
    drop_cols=None,
    optional_drop_cols=None,
    n_splits=5,
    param_grid=None,
):
    if drop_cols is None:
        drop_cols = [target_col, date_col, group_col]
    if optional_drop_cols is None:
        optional_drop_cols = ["lightning_id"]

    if param_grid is None:
        param_grid = {
            "n_estimators": [200, 400],
            "learning_rate": [0.03, 0.05],
            "num_leaves": [15, 31],
            "max_depth": [6, -1],
            "min_child_samples": [20, 50],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "reg_alpha": [0.0, 0.5],
            "reg_lambda": [0.0, 1.0],
            "class_weight": [None, "balanced"],
        }

    df_cv = model_df_train.copy()
    df_cv = df_cv[df_cv[date_col].notna()].reset_index(drop=True)

    X = df_cv.drop(
        columns=[c for c in drop_cols + optional_drop_cols if c in df_cv.columns],
        errors="ignore"
    ).copy()
    y = df_cv[target_col].astype(int).copy()

    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    for c in cat_cols:
        X[c] = X[c].astype("category")

    folds = make_purged_time_folds(
        df=df_cv,
        n_splits=n_splits,
        date_col=date_col,
        group_col=group_col,
    )

    print(f"Nombre de folds utilisables : {len(folds)}")

    results = []
    best_score = -np.inf
    best_params = None

    all_params = list(ParameterGrid(param_grid))
    print(f"Nombre de combinaisons testées : {len(all_params)}")

    for i, params in enumerate(all_params, 1):
        fold_auc = []
        fold_ap = []
        fold_brier = []

        print(f"\n[{i}/{len(all_params)}] Params = {params}")

        for fold_id, (train_idx, val_idx) in enumerate(folds, 1):
            X_tr = X.iloc[train_idx].copy()
            y_tr = y.iloc[train_idx].copy()
            X_va = X.iloc[val_idx].copy()
            y_va = y.iloc[val_idx].copy()

            for c in cat_cols:
                if c in X_tr.columns:
                    X_tr[c] = X_tr[c].astype("category")
                if c in X_va.columns:
                    X_va[c] = X_va[c].astype("category")

            model = LGBMClassifier(
                objective="binary",
                random_state=42,
                n_jobs=-1,
                **params
            )

            model.fit(
                X_tr,
                y_tr,
                categorical_feature=cat_cols if len(cat_cols) > 0 else "auto",
            )

            p_va = model.predict_proba(X_va)[:, 1]

            auc = roc_auc_score(y_va, p_va)
            ap = average_precision_score(y_va, p_va)
            brier = brier_score_loss(y_va, p_va)

            fold_auc.append(auc)
            fold_ap.append(ap)
            fold_brier.append(brier)

            n_train_alerts = df_cv.iloc[train_idx][group_col].nunique()
            n_val_alerts = df_cv.iloc[val_idx][group_col].nunique()

            print(
                f"  Fold {fold_id} | "
                f"AUC={auc:.4f} | AP={ap:.4f} | Brier={brier:.4f} | "
                f"n_train={len(train_idx)} | n_val={len(val_idx)} | "
                f"alerts_train={n_train_alerts} | alerts_val={n_val_alerts}"
            )

        mean_auc = float(np.mean(fold_auc))
        mean_ap = float(np.mean(fold_ap))
        mean_brier = float(np.mean(fold_brier))

        results.append({
            **params,
            "mean_auc": mean_auc,
            "mean_ap": mean_ap,
            "mean_brier": mean_brier,
        })

        print(
            f"--> Moyennes | AUC={mean_auc:.4f} | AP={mean_ap:.4f} | Brier={mean_brier:.4f}"
        )

        score = mean_ap + 0.1 * mean_auc - 0.05 * mean_brier

        if score > best_score:
            best_score = score
            best_params = params.copy()

    results_df = pd.DataFrame(results).sort_values(
        ["mean_ap", "mean_auc", "mean_brier"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    return best_params, results_df

In [4]:
t0 = time.time()

df_train_feat, strong_cg_threshold = add_base_columns(df_train, strong_cg_threshold=None)
df_valid_feat, _ = add_base_columns(df_valid, strong_cg_threshold=strong_cg_threshold)

# TRAIN
base_train = build_timestamp_features(df_train_feat)
base_time_train = add_since_features(base_train)
features_train = add_rolling_features(base_time_train)
features_train = add_derived_features(features_train)

# VALID
base_valid = build_timestamp_features(df_valid_feat)
base_time_valid = add_since_features(base_valid)
features_valid = add_rolling_features(base_time_valid)
features_valid = add_derived_features(features_valid)

print("Feature build time:", round(time.time() - t0, 2), "s")
print("features_train:", features_train.shape)
print("features_valid:", features_valid.shape)

target_train, _ = compute_target_safe_to_end(df_train_feat, danger_dist_km=20)
alert_life_train = compute_alert_life(df_train_feat)
airport_static = compute_airport_static(df_train_feat)

model_df_train = (
    target_train
    .merge(features_train, on=["airport", "date"], how="left")
    .merge(alert_life_train, on=["airport", "airport_alert_id", "date"], how="left")
    .merge(airport_static, on="airport", how="left")
)

model_df_train["airport"] = model_df_train["airport"].astype("category")
if "season" in model_df_train.columns:
    model_df_train["season"] = model_df_train["season"].astype("category")

print(model_df_train.shape)
print(model_df_train["safe_to_end_now"].mean())

Feature build time: 21.23 s
features_train: (400210, 146)
features_valid: (168170, 146)
(53753, 163)
0.048871690882369354


In [5]:
target_valid, _ = compute_target_safe_to_end(df_valid_feat, danger_dist_km=20)
alert_life_valid = compute_alert_life(df_valid_feat)

test_candidates = (
    df_valid_feat[df_valid_feat["airport_alert_id"].notna()][["airport", "airport_alert_id", "date"]]
    .drop_duplicates()
    .copy()
)

model_df_valid = (
    test_candidates
    .merge(features_valid, on=["airport", "date"], how="left")
    .merge(alert_life_valid, on=["airport", "airport_alert_id", "date"], how="left")
    .merge(airport_static, on="airport", how="left")
)

model_df_valid["airport"] = model_df_valid["airport"].astype("category")
if "season" in model_df_valid.columns:
    model_df_valid["season"] = model_df_valid["season"].astype("category")

print(model_df_valid.shape)

TARGET_COL = "safe_to_end_now"

DROP_COLS = [
    TARGET_COL,
    "date",
    "airport_alert_id",
]

OPTIONAL_DROP_COLS = [
    "lightning_id",
]

X_train = model_df_train.drop(columns=[c for c in DROP_COLS + OPTIONAL_DROP_COLS if c in model_df_train.columns], errors="ignore").copy()
y_train = model_df_train[TARGET_COL].astype(int).copy()

(13298, 162)


In [ ]:
param_grid = {
    "n_estimators": [200, 400, 700],
    "learning_rate": [0.03, 0.05],
    "num_leaves": [15, 31],
    "max_depth": [6, -1],
    "min_child_samples": [20, 50],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_alpha": [0.0],
    "reg_lambda": [0.0, 1.0],
    "class_weight": [None, "balanced"],
}

best_params_purged, cv_results_purged = temporal_purged_cv_hyperparam_search(
    model_df_train=model_df_train,
    target_col="safe_to_end_now",
    date_col="date",
    group_col="airport_alert_id",
    drop_cols=DROP_COLS,
    optional_drop_cols=OPTIONAL_DROP_COLS,
    n_splits=5,
    param_grid=param_grid,
)

print("\nMeilleurs hyperparamètres (CV temporelle purgée) :")
print(best_params_purged)

print("\nTop 10 résultats :")
print(cv_results_purged.head(10))

cv_results_purged.to_csv("cv_results_lgbm_temporal_purged.csv", index=False)

Nombre de folds utilisables : 5
Nombre de combinaisons testées : 192

[1/192] Params = {'class_weight': None, 'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 6, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 15, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.8}
[LightGBM] [Info] Number of positive: 242, number of negative: 2894
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002140 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25142
[LightGBM] [Info] Number of data points in the train set: 3136, number of used features: 151
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.077168 -> initscore=-2.481457
[LightGBM] [Info] Start training from score -2.481457
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g